# VERA: conservative Qwen3.5-4B pilot on two T4 GPUs
Attach `vera-domain-v1.zip` and the locally generated `vera-kaggle-training.zip` as private Kaggle inputs. Select **GPU T4 x2** and enable Internet. Read `docs/kaggle-finetuning.md` first.
This notebook uses single-process **FP32 LoRA with model sharding**, not DDP, FP16, BF16, QLoRA, or GGUF training. Qwen3.5's FP16 stability and T4 memory fit must not be assumed. The first experiment performs five optimizer updates, including worst-case token lengths. It is not evidence of improved model quality.
The dataset remains a synthetic, unapproved candidate. No training starts until you explicitly acknowledge its limitations. The full run is disabled by default.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile


def unique_input(name):
    matches = list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one {name}; found {len(matches)}')
    return matches[0]

DATASET = unique_input('vera-domain-v1.zip')
BUNDLE = unique_input('vera-kaggle-training.zip')
WORK = Path('/kaggle/working')
CODE = WORK / 'vera-training-code'
CODE.mkdir(exist_ok=True)
modules = ('__init__', 'data', 'preflight', 'model', 'checkpoints', 'train')
allowed = {'requirements-kaggle.txt', 'runtime-contract.json', 'LICENSE'}
allowed.update(f'vera_finetuning/{name}.py' for name in modules)
with ZipFile(BUNDLE) as archive:
    if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
        raise RuntimeError('Unexpected files in the training bundle')
    if sum(item.file_size for item in archive.infolist()) > 1024 * 1024:
        raise RuntimeError('Training code bundle is unexpectedly large')
    archive.extractall(CODE)
CONTRACT = CODE / 'runtime-contract.json'
ENV = {**os.environ, 'PYTHONPATH': str(CODE), 'HF_HOME': str(WORK / 'hf-cache')}
print(json.loads(CONTRACT.read_text()))

In [ ]:
VENV = WORK / 'vera-training-venv'
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r',
                str(CODE / 'requirements-kaggle.txt')], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'], check=True)

In [ ]:
COMMON = ['--dataset', str(DATASET), '--contract', str(CONTRACT)]
subprocess.run([PYTHON, '-m', 'vera_finetuning.preflight', *COMMON,
                '--output', str(WORK / 'vera-preflight.json')], env=ENV, check=True)

## Review the preflight before continuing
Verify the split counts, token limits, assistant-only boundary and warnings. An inconsistent historical manifest prompt hash is reported separately; every actual conversation must match the captured runtime prompt. A changed runtime, including the `chat` branch's prompt, requires a rebuilt and reviewed dataset.
Set `ACKNOWLEDGE_CANDIDATE_DATA = True` only after reviewing those limitations. The pilot may stop with an out-of-memory or kernel error: retain the error and do not silently truncate examples or switch to unstable reduced precision.

In [ ]:
ACKNOWLEDGE_CANDIDATE_DATA = False
RUN_FULL_TRAINING = False
RUN_TAG = datetime.now(UTC).strftime('%Y%m%d-%H%M%S')
PILOT_OUTPUT = WORK / f'vera-pilot-{RUN_TAG}'
FULL_OUTPUT = WORK / f'vera-full-{RUN_TAG}'
FULL_RESUME = None

if not ACKNOWLEDGE_CANDIDATE_DATA:
    raise RuntimeError('Review the candidate-data warnings and explicitly acknowledge them first')
subprocess.run([PYTHON, '-m', 'vera_finetuning.train', *COMMON, '--pilot',
                '--output', str(PILOT_OUTPUT), '--acknowledge-candidate-data'], env=ENV, check=True)
pilot_report = json.loads((PILOT_OUTPUT / 'report.json').read_text())
if pilot_report['steps'] != 5:
    raise RuntimeError('The pilot did not complete all five optimizer updates')
print(json.dumps(pilot_report, indent=2))

In [ ]:
if RUN_FULL_TRAINING:
    if not ACKNOWLEDGE_CANDIDATE_DATA or pilot_report['steps'] != 5:
        raise RuntimeError('Explicit consent and a successful pilot are required')
    command = [PYTHON, '-m', 'vera_finetuning.train', *COMMON, '--full',
               '--output', str(FULL_OUTPUT), '--acknowledge-candidate-data']
    if FULL_RESUME is not None:
        command += ['--resume', str(FULL_RESUME)]
    subprocess.run(command, env=ENV, check=True)
else:
    print('Full training is disabled. A successful pilot does not establish improved quality.')

## Preserve outputs and evaluate before deployment
Use Kaggle **Save Version** and retain the output directories. Each contains `report.json`, an `adapter/` directory and complete checkpoints. Resume a full run from a saved `checkpoint-*` directory using a new output directory.
Four validation losses are only a numerical smoke check, not a quality benchmark. Compare the base and adapted model on matched validation contexts, verify JSON, citations, missing-data behavior, scope and human-review behavior; reserve the test split for the final comparison. Never deploy based only on decreasing loss.
The adapter is not a GGUF and cannot replace the existing MTP GGUF file. Merge it with the same pinned Hugging Face base checkpoint, convert using a compatible llama.cpp version, and repeat VERA's runtime checks before selecting it.